# 01 — Data Profiling: Indonesia News

Notebook ini melakukan profiling awal dataset [Indonesia News](https://www.kaggle.com/datasets/azizainunnajib/indonesia-news) untuk Milestone 1. Dataset berisi artikel dari CNN Indonesia dan Tempo dalam tiga file CSV dengan ukuran total lebih dari 2 GB.

Tujuan profiling:

- memverifikasi ukuran dan struktur file;
- menghitung jumlah baris dan rentang waktu;
- memeriksa nilai null atau kosong;
- mengidentifikasi potensi duplikasi;
- melihat distribusi sumber, panjang artikel, dan tag awal.

> Data mentah hanya dibaca dengan Polars lazy API dan tidak diubah.

In [ ]:
from pathlib import Path

import polars as pl

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(100)

print(f"Polars version: {pl.__version__}")

## 1. Lokasi dan konfigurasi file

Dua file menggunakan pemisah `|`, sedangkan `tempo010619310520.csv` menggunakan `;`. Konfigurasi dipisahkan per file agar pembacaan data tetap konsisten.

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "archive"

FILE_CONFIG = [
    {"name": "cnn.csv", "separator": "|", "source": "CNN Indonesia"},
    {"name": "tempo.csv", "separator": "|", "source": "Tempo"},
    {"name": "tempo010619310520.csv", "separator": ";", "source": "Tempo"},
]

missing_files = [item["name"] for item in FILE_CONFIG if not (DATA_DIR / item["name"]).exists()]
if missing_files:
    raise FileNotFoundError(
        f"File belum ditemukan di {DATA_DIR}: {', '.join(missing_files)}"
    )

print(f"Folder data: {DATA_DIR}")

## 2. Inventaris dan ukuran data

Ukuran dihitung langsung dari file hasil ekstraksi. Syarat tugas terpenuhi apabila total data minimal 500 MB atau jumlah baris lebih dari satu juta.

In [ ]:
inventory = pl.DataFrame(
    [
        {
            "file": item["name"],
            "source": item["source"],
            "separator": item["separator"],
            "size_bytes": (DATA_DIR / item["name"]).stat().st_size,
        }
        for item in FILE_CONFIG
    ]
).with_columns(
    (pl.col("size_bytes") / 1024**2).round(2).alias("size_mib"),
    (pl.col("size_bytes") / 1024**3).round(3).alias("size_gib"),
)

total_bytes = inventory["size_bytes"].sum()
display(inventory)
print(f"Total ukuran: {total_bytes / 1024**3:.3f} GiB")
print(f"Memenuhi batas minimal 500 MB: {total_bytes >= 500_000_000}")

## 3. Membaca data secara lazy

`scan_csv` digunakan agar Polars menyusun rencana query terlebih dahulu. Kolom waktu dinormalisasi dari teks atau notasi ilmiah menjadi Unix timestamp milidetik, kemudian dikonversi menjadi `Datetime`.

In [ ]:
EXPECTED_COLUMNS = ["content", "datetime", "tags", "title"]

def scan_news_file(config: dict) -> pl.LazyFrame:
    path = DATA_DIR / config["name"]
    datetime_ms = (
        pl.col("datetime")
        .cast(pl.String)
        .str.replace_all(",", ".")
        .cast(pl.Float64, strict=False)
        .round(0)
        .cast(pl.Int64, strict=False)
    )

    return (
        pl.scan_csv(
            path,
            separator=config["separator"],
            has_header=True,
            quote_char='"',
            encoding="utf8-lossy",
            infer_schema_length=10_000,
            schema_overrides={column: pl.String for column in EXPECTED_COLUMNS},
            null_values=["", "null", "None", "NA", "N/A"],
            ignore_errors=True,
            truncate_ragged_lines=True,
            low_memory=True,
        )
        .select(EXPECTED_COLUMNS)
        .with_columns(
            pl.lit(config["source"]).alias("source"),
            pl.lit(config["name"]).alias("source_file"),
            datetime_ms.alias("datetime_ms"),
        )
        .with_columns(
            pl.from_epoch(pl.col("datetime_ms"), time_unit="ms").alias("published_at")
        )
    )

lazy_frames = [scan_news_file(item) for item in FILE_CONFIG]
news = pl.concat(lazy_frames, how="vertical_relaxed")

news.collect_schema()

## 4. Sampel data

Sampel digunakan untuk memeriksa apakah delimiter, judul, isi, tag, dan tanggal telah dibaca dengan benar.

In [ ]:
sample = (
    news.select(
        "source",
        "source_file",
        "published_at",
        "title",
        "tags",
        pl.col("content").str.slice(0, 250).alias("content_preview"),
    )
    .head(10)
    .collect(streaming=True)
)
sample

## 5. Jumlah baris dan rentang waktu

Bagian ini memverifikasi batas skala data sekaligus periode aktual yang tersedia.

In [ ]:
overall_summary = news.select(
    pl.len().alias("row_count"),
    pl.col("title").n_unique().alias("unique_titles"),
    pl.col("published_at").is_not_null().sum().alias("valid_datetime_rows"),
    pl.col("published_at").min().alias("earliest_datetime"),
    pl.col("published_at").max().alias("latest_datetime"),
).collect(streaming=True)

overall_summary

In [ ]:
source_summary = (
    news.group_by("source", "source_file")
    .agg(
        pl.len().alias("row_count"),
        pl.col("title").n_unique().alias("unique_titles"),
        pl.col("published_at").min().alias("earliest_datetime"),
        pl.col("published_at").max().alias("latest_datetime"),
    )
    .sort("row_count", descending=True)
    .collect(streaming=True)
)

source_summary

## 6. Nilai null dan string kosong

Nilai null dan string kosong dihitung bersama karena keduanya menunjukkan informasi yang tidak tersedia.

In [ ]:
PROFILE_COLUMNS = ["content", "datetime", "tags", "title", "published_at"]

missing_summary = news.select(
    [
        (
            pl.col(column).null_count()
            + pl.col(column).cast(pl.String).str.strip_chars().eq("").sum()
        ).alias(column)
        for column in PROFILE_COLUMNS
    ]
).collect(streaming=True)

missing_summary.transpose(
    include_header=True,
    header_name="column",
    column_names=["missing_or_empty"],
)

## 7. Potensi duplikasi

Duplikasi awal dinilai dari kombinasi sumber dan judul. Pemeriksaan yang lebih ketat terhadap isi artikel dilakukan pada notebook cleaning.

In [ ]:
duplicate_summary = news.select(
    pl.len().alias("row_count"),
    pl.struct(["source", "title"]).n_unique().alias("unique_source_title"),
).with_columns(
    (pl.col("row_count") - pl.col("unique_source_title")).alias("potential_duplicate_rows")
).collect(streaming=True)

duplicate_summary

## 8. Profil panjang teks

Panjang judul dan artikel membantu menemukan teks kosong, terlalu pendek, atau nilai ekstrem yang perlu ditangani pada tahap cleaning.

In [ ]:
text_profile = (
    news.with_columns(
        pl.col("title").str.len_chars().alias("title_length"),
        pl.col("content").str.len_chars().alias("content_length"),
    )
    .select(
        pl.col("title_length").min().alias("title_min"),
        pl.col("title_length").median().alias("title_median"),
        pl.col("title_length").mean().round(2).alias("title_mean"),
        pl.col("title_length").max().alias("title_max"),
        pl.col("content_length").min().alias("content_min"),
        pl.col("content_length").median().alias("content_median"),
        pl.col("content_length").mean().round(2).alias("content_mean"),
        pl.col("content_length").max().alias("content_max"),
        (pl.col("content_length") < 100).sum().alias("content_under_100_chars"),
    )
    .collect(streaming=True)
)

text_profile

## 9. Tag yang paling sering muncul

Nilai tag masih diperlakukan sebagai teks mentah. Pemisahan tag individual dilakukan pada tahap cleaning setelah formatnya diperiksa.

In [ ]:
top_raw_tags = (
    news.filter(pl.col("tags").is_not_null() & pl.col("tags").str.strip_chars().ne(""))
    .group_by("tags")
    .agg(pl.len().alias("article_count"))
    .sort("article_count", descending=True)
    .head(20)
    .collect(streaming=True)
)

top_raw_tags

## 10. Catatan hasil profiling

Setelah seluruh sel dijalankan, isi bagian berikut berdasarkan hasil yang tampil:

1. **Skala data:** tuliskan jumlah baris dan ukuran total.
2. **Periode:** tuliskan tanggal paling awal dan paling akhir.
3. **Kelengkapan:** sebutkan kolom dengan null atau string kosong terbanyak.
4. **Duplikasi:** tuliskan jumlah baris yang berpotensi duplikat berdasarkan sumber dan judul.
5. **Kualitas teks:** jelaskan artikel yang terlalu pendek atau panjang secara ekstrem.
6. **Rencana cleaning:** jelaskan tindakan yang akan dilakukan pada notebook `02_data_cleaning.ipynb`.